<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signals

- `age_days`: number of days from the last update to the decision date.


### Score

`score = w1 × age_norm + w2 × decline_norm`

In [1]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login

TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY   = f"{BASE}/fact_content_daily_performance/**/*.parquet"
SRC_CLIENTS = f"{BASE}/dim_clients.parquet"
SRC_CONTENT = f"{BASE}/dim_content.parquet"

START, END  = "2026-03-01", "2026-03-31"
MIN_IMPR_M3 = 100   # الحد الأدنى للـ impressions في شهر 3

con.execute(f"""
CREATE OR REPLACE TEMP TABLE content_slice AS
SELECT d.client_hash_id, d.content_hash_id
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN read_parquet('{SRC_CLIENTS}') c USING (client_hash_id)
WHERE d.month = '2026-03'
  AND d.report_date BETWEEN DATE '{START}' AND DATE '{END}'
  AND CAST(c.gsc_data_start AS DATE) < DATE '{START}'
GROUP BY d.client_hash_id, d.content_hash_id
HAVING SUM(d.gsc_impressions) >= {MIN_IMPR_M3}
""")

slice_df = con.execute("SELECT * FROM content_slice").df()

print(len(slice_df))
print(slice_df)


HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101168
                 client_hash_id           content_hash_id
0       client_73cda7b4e4f265ea  content_b7e512995f79d5a6
1       client_73cda7b4e4f265ea  content_905aa32a0230694e
2       client_73cda7b4e4f265ea  content_05434271b257bb68
3       client_73cda7b4e4f265ea  content_d056587ff7faca0c
4       client_73cda7b4e4f265ea  content_2662845f598544ef
...                         ...                       ...
101163  client_3ffa76342f366962  content_48c9b57be92bb11b
101164  client_3ffa76342f366962  content_52c98b3f26951f3b
101165  client_1a730cb2640a1abf  content_051c8125a3d20e83
101166  client_1a730cb2640a1abf  content_45155b648fff917d
101167  client_b10cb2997d0c7c86  content_2238789058b6c875

[101168 rows x 2 columns]


In [2]:
stale_df = con.execute(f"""
SELECT
    d.content_hash_id AS content,
    s.client_hash_id AS client,
    DATE_DIFF(
        'day',
        CAST(
            CASE
                WHEN d.content_updated_date IS NOT NULL
                     AND d.content_updated_date <= DATE '2026-03-31'
                THEN d.content_updated_date
                ELSE d.content_created_date
            END AS DATE
        ),
        DATE '2026-03-31'
    ) AS stale_days
FROM read_parquet('{SRC_CONTENT}') d
JOIN slice_df s USING (content_hash_id)
WHERE CASE
    WHEN d.content_updated_date IS NOT NULL
         AND d.content_updated_date <= DATE '2026-03-31'
    THEN d.content_updated_date
    ELSE d.content_created_date
END IS NOT NULL
""").df()



ctr_peers_df = con.execute(f"""
WITH page_data AS (
    SELECT
        d.content_hash_id AS content,
        d.client_hash_id AS client,
        SUM(d.gsc_clicks) * 1.0
            / NULLIF(SUM(d.gsc_impressions), 0) AS ctr,
        AVG(d.gsc_avg_position) AS avg_position
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN slice_df s USING (content_hash_id, client_hash_id)
    WHERE d.month = '2026-03'
      AND d.report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY d.content_hash_id, d.client_hash_id
),
peer_data AS (
    SELECT
        content,
        client,
        ctr,
        CASE
            WHEN avg_position <= 3 THEN 'top_3'
            WHEN avg_position <= 10 THEN 'page_1'
            WHEN avg_position <= 20 THEN 'striking'
            WHEN avg_position <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS position_tier
    FROM page_data
),
peer_average AS (
    SELECT
        content,
        client,
        ctr,
        (
            SUM(ctr) OVER (PARTITION BY position_tier) - ctr
        ) / NULLIF(
            COUNT(*) OVER (PARTITION BY position_tier) - 1, 0
        ) AS peers_avg_ctr
    FROM peer_data
)
SELECT
    content,
    client,
    ctr / NULLIF(peers_avg_ctr, 0) AS ctr_per_peers
FROM peer_average
WHERE ctr IS NOT NULL
  AND peers_avg_ctr IS NOT NULL
  AND peers_avg_ctr > 0
""").df()



label_df = con.execute(f"""
WITH monthly AS (
    SELECT
        d.content_hash_id AS content,
        d.client_hash_id AS client,
        AVG(CASE WHEN d.month = '2026-03'
                 THEN d.gsc_impressions END) AS avg_march,
        AVG(CASE WHEN d.month = '2026-04'
                 THEN d.gsc_impressions END) AS avg_april
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN slice_df s USING (content_hash_id, client_hash_id)
    WHERE d.month IN ('2026-03', '2026-04')
    GROUP BY d.content_hash_id, d.client_hash_id
)
SELECT
    content,
    client,
    CASE
        WHEN avg_march IS NULL OR avg_march = 0 OR avg_april IS NULL
            THEN NULL
        WHEN (avg_april - avg_march) / avg_march > 0.20
            THEN 'Up'
        WHEN (avg_april - avg_march) / avg_march < -0.20
            THEN 'Down'
        ELSE 'Stable'
    END AS label
FROM monthly
WHERE avg_march IS NOT NULL
  AND avg_march > 0
  AND avg_april IS NOT NULL
""").df()



print(stale_df)
print(ctr_peers_df)
print(label_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                         content                   client  stale_days
0       content_b7e512995f79d5a6  client_73cda7b4e4f265ea         396
1       content_905aa32a0230694e  client_73cda7b4e4f265ea         396
2       content_05434271b257bb68  client_73cda7b4e4f265ea         396
3       content_d056587ff7faca0c  client_73cda7b4e4f265ea         396
4       content_2662845f598544ef  client_73cda7b4e4f265ea         396
...                          ...                      ...         ...
101163  content_48c9b57be92bb11b  client_3ffa76342f366962         425
101164  content_52c98b3f26951f3b  client_3ffa76342f366962         245
101165  content_051c8125a3d20e83  client_1a730cb2640a1abf           3
101166  content_45155b648fff917d  client_1a730cb2640a1abf           1
101167  content_2238789058b6c875  client_b10cb2997d0c7c86           1

[101168 rows x 3 columns]
                         content                   client  ctr_per_peers
0       content_243b7a6bb1a1fd16  client_73cda7b4e4f265ea   

In [3]:
# Stale summary
stale_analysis = label_df.merge(stale_df, on=["content", "client"])

stale_analysis["stale_range"] = pd.cut(
    stale_analysis["stale_days"],
    bins=[-float("inf"), 30, 90, 180, float("inf")],
    labels=["0-30", "31-90", "91-180", "180+"]
)

stale_counts = stale_analysis.groupby(
    ["stale_range", "label"], observed=False
).size().unstack(fill_value=0)

stale_counts = stale_counts.reindex(
    columns=["Down", "Up", "Stable"], fill_value=0
)

stale_summary = stale_counts.div(
    stale_counts.sum(axis=1).replace(0, float("nan")), axis=0
).mul(100).round(1)

stale_summary["Count"] = stale_counts.sum(axis=1)
stale_summary = stale_summary.reset_index()

display(stale_summary)


# CTR per peers summary
peers_analysis = label_df.merge(
    ctr_peers_df, on=["content", "client"]
)

peers_analysis["ctr_peers_range"] = pd.cut(
    peers_analysis["ctr_per_peers"],
    bins=[-float("inf"), 0.5, 0.8, 1.2, float("inf")],
    labels=["Very Low", "Low", "Average", "High"]
)

peers_counts = peers_analysis.groupby(
    ["ctr_peers_range", "label"], observed=False
).size().unstack(fill_value=0)

peers_counts = peers_counts.reindex(
    columns=["Down", "Up", "Stable"], fill_value=0
)

peers_summary = peers_counts.div(
    peers_counts.sum(axis=1).replace(0, float("nan")), axis=0
).mul(100).round(1)

peers_summary["Count"] = peers_counts.sum(axis=1)
peers_summary = peers_summary.reset_index()

display(peers_summary)

label,stale_range,Down,Up,Stable,Count
0,0-30,33.3,48.4,18.3,8415
1,31-90,51.6,24.3,24.1,38574
2,91-180,58.9,16.7,24.4,11849
3,180+,50.1,23.3,26.5,42330


label,ctr_peers_range,Down,Up,Stable,Count
0,Very Low,58.0,19.8,22.3,52279
1,Low,51.4,21.4,27.1,10507
2,Average,47.2,24.5,28.3,10292
3,High,36.8,36.3,27.0,28090


In [5]:
import numpy as np

COOLDOWN_DAYS = 30
STALE_DAYS    = 180
CTR_RATIO     = 0.5

rule_df = stale_df.merge(ctr_peers_df, on=["content", "client"])


rule_df["age_norm"]     = rule_df["stale_days"].rank(pct=True)
rule_df["ctr_gap_norm"] = 1 - rule_df["ctr_per_peers"].rank(pct=True)

# score
rule_df["score"] = rule_df["age_norm"] * rule_df["ctr_gap_norm"]

# reason code + action
cooldown = rule_df["stale_days"] <= COOLDOWN_DAYS
low_ctr  = rule_df["ctr_per_peers"] < CTR_RATIO
stale    = rule_df["stale_days"] > STALE_DAYS

conds   = [cooldown, low_ctr & stale, low_ctr, stale]
reasons = ["WITHIN_COOLDOWN", "CTR_BELOW_POSITION_PEERS + STALE_CONTENT", "CTR_BELOW_POSITION_PEERS", "STALE_CONTENT"]
actions = ["monitor", "snippet_fix + refresh", "snippet_fix", "refresh"]

rule_df["reason_code"] = np.select(conds, reasons, default="")
rule_df["action"]      = np.select(conds, actions, default="")

rule_df = rule_df.sort_values("score", ascending=False)
rule_df[["content", "client", "stale_days", "ctr_per_peers", "score", "reason_code", "action"]].head(10)

,content,client,stale_days,ctr_per_peers,score,reason_code,action
4283,content_609e63e1b1f6fba1,client_c182d11e4862a37d,494,0.0,0.813795,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
4282,content_f657a2259072ff88,client_c182d11e4862a37d,494,0.0,0.813795,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
54935,content_686064cfde37e2b3,client_c182d11e4862a37d,494,0.0,0.813795,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
4068,content_4f41372ca054fe55,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
54930,content_3260912233fa2850,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
54742,content_43975e7a1b05ab1b,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
54741,content_94c62a8802a50142,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
54739,content_31f3d461b02c357f,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
54738,content_4a1a5c83f617267e,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
54737,content_c1cec58795992942,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh


In [ ]:
KS = [200, 500, 1000, 2000, 50500]

eval_df = rule_df.merge(label_df, on=["content", "client"])
eval_df = eval_df.sort_values("score", ascending=False)

base_rate = (eval_df["label"] == "Down").mean()
print(f"base rate (Down): {base_rate:.1%}")

for K in KS:
    top_k = eval_df.head(K)
    precision_at_k = (top_k["label"] == "Down").mean()
    print(f"precision@{K}: {precision_at_k:.1%}")



base rate (Down): 50.3%
precision@200: 94.5%
precision@500: 65.4%
precision@1000: 66.2%
precision@2000: 67.5%
precision@50500: 55.0%


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import base64, requests
from getpass import getpass

TOKEN = getpass("GitHub token: ")
REPO  = "ahmedabdelshaheed/flyrank_machinelearning"
PATH  = "work/outputs/baseline_action_score.csv"

url = f"https://api.github.com/repos/{REPO}/contents/{PATH}"
headers = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}

with open(PATH, "rb") as f:
    content = base64.b64encode(f.read()).decode()

payload = {"message": "Add baseline_action_score.csv", "content": content, "branch": "main"}

# لو الملف موجود، بنحتاج sha عشان نحدّثه
r = requests.get(url, headers=headers)
if r.status_code == 200:
    payload["sha"] = r.json()["sha"]

r = requests.put(url, headers=headers, json=payload)
print(r.status_code)
print(r.json().get("content", {}).get("html_url", r.text[:300]))

GitHub token: ··········
201
https://github.com/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
URL = "https://raw.githubusercontent.com/ahmedabdelshaheed/flyrank_machinelearning/main/work/outputs/baseline_action_score.csv"

check = pd.read_csv(URL)
print(check.shape)
check.head(20)

(101168, 8)


,rank,content,client,stale_days,ctr_per_peers,score,reason_code,action
0,1,content_609e63e1b1f6fba1,client_c182d11e4862a37d,494,0.0,0.813795,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
1,2,content_f657a2259072ff88,client_c182d11e4862a37d,494,0.0,0.813795,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
2,3,content_686064cfde37e2b3,client_c182d11e4862a37d,494,0.0,0.813795,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
3,4,content_4f41372ca054fe55,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
4,5,content_3260912233fa2850,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
5,6,content_43975e7a1b05ab1b,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
6,7,content_94c62a8802a50142,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
7,8,content_31f3d461b02c357f,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
8,9,content_4a1a5c83f617267e,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
9,10,content_c1cec58795992942,client_c182d11e4862a37d,487,0.0,0.811857,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.